# LAB04 + LAB05 · Del JSON al Parquet
## Cierre del Bloque 1

**Curso Big Data e IA Aplicada · Formación San Miguel**

---

Hoy cierras el bloque tocando los dos extremos que te faltaban:

- **Hacia arriba**, el **JSON**: el idioma de las APIs, que no cabe en una tabla plana.
- **Hacia abajo**, el **almacenamiento**: cómo se reparte un fichero entre máquinas (HDFS) y cómo
  se guarda de verdad en la industria (Parquet).

Y produces **dos artefactos que el curso necesita de verdad**:

| Artefacto | Para qué |
|---|---|
| `productos.csv` | La semana que viene se une a las ventas con un **JOIN** |
| `ventas.parquet` | Es el formato sobre el que trabajará **todo** el bloque 2 |

---

### Las cuatro etiquetas — las mismas de ayer

| Etiqueta | A quién preguntas | Para qué |
|---|---|---|
| 🗂️ **RAG** | NotebookLM, con el material del curso | Conceptos. **Exígele que cite el apartado** |
| 🤖 **ASISTENTE** | Gemini o ChatGPT | Sintaxis, con el protocolo de cuatro pasos |
| ⚙️ **MÁQUINA** | A nadie: **se ejecuta** | Los números |
| 📝 **CRITERIO** | A nadie: **lo escribes tú** | Lo único que la IA no puede poner |

> **La regla:** si la respuesta es un número de tus datos, no se le pregunta a ninguna IA. **Se
> ejecuta.**

### El mapa de los sitios — hoy importa más que nunca

| Dónde | Qué corre ahí |
|---|---|
| `▸` **Terminal de JupyterLab** | `jq`, y todo lo del LAB04 |
| `◉` **Celda de cuaderno** | DuckDB y las medidas del LAB05 |
| `⌂` **Terminal del puesto (ssh)** | **HDFS.** Desde aquí NO se puede: un contenedor no gobierna a sus hermanos |

Este cuaderno vive en `notebooks/`. Los datos, un piso arriba: `../datasets/`

## 0 · Comprobación del entorno

`jq` **no viene** en la imagen. Si falta, la celda te dice cómo instalarlo.

In [ ]:
import os, shutil

print("Estoy en:", os.getcwd())
print()

for f in ["ventas.csv", "productos.json", "clientes.csv", "access.log"]:
    ruta = f"../datasets/{f}"
    if os.path.exists(ruta):
        print(f"  OK    {f:16s} {os.path.getsize(ruta)/1024/1024:7.1f} MB")
    else:
        print(f"  FALTA {f}  <-- avisa al docente")

print()
if shutil.which("jq"):
    print("  OK    jq disponible")
else:
    print("  FALTA jq  ->  en la Terminal:  mamba install -y -c conda-forge jq")

try:
    import duckdb
    print("  OK    duckdb", duckdb.__version__)
except ImportError:
    print("  FALTA duckdb  ->  en la Terminal:  pip install duckdb   (y reinicia el kernel)")

---
---

# LAB04 · jq: del JSON al CSV

## Por qué el JSON necesita otra herramienta

Un CSV y un log piensan en **líneas y columnas**. Un JSON no es ni lo uno ni lo otro: es una
**estructura jerárquica** donde un registro puede contener listas y objetos anidados.

Mira un producto del catálogo:

```json
{ "id": "P-1042", "nombre": "Portátil 15.6 i5", "categoria": "informatica",
  "precio": 219.90, "tags": ["ofimatica", "portatil"],
  "stock": { "central": 14, "tiendas": 6 } }
```

Fíjate en `stock`: **un objeto dentro del producto**, con dos niveles. Eso es exactamente lo que una
tabla plana no sabe expresar — y la razón de que este formato domine el intercambio entre servicios.

## Las cinco piezas de jq

| Pieza | Qué hace |
|---|---|
| `.` | El documento entero |
| `.campo` | Baja un nivel. Se encadena: `.stock.central` baja dos |
| `.[]` | Itera: aplica lo que sigue a cada elemento de la lista |
| <code>&#124;</code> | Encadena filtros — **la misma tubería del shell, dentro del JSON** |
| `select()` | Filtra por condición: el `WHERE` de jq |

---

## Paso 1 · Reconocer el terreno

In [ ]:
%%bash
cd ../datasets

echo "== ¿Cuántos productos hay? =="
jq 'length' productos.json

echo ""
echo "== El primer producto, completo =="
jq '.[0]' productos.json

echo ""
echo "== Sus campos =="
jq '.[0] | keys' productos.json

# Esperado: 480 productos
# El primero es P-0887 «Edredón nórdico Uno», 298.68, con tags y un stock de dos niveles.

> 📌 **Anota el `length`.** Es contra lo que se valida el paso siguiente.
>
> *Todo artefacto se valida contra algo. Este se valida contra su origen.*

---

## Paso 2 · La transformación, en dos tiempos

### Tiempo A · La idea desnuda

Cada producto se convierte en una **lista de valores**, y `@csv` escribe esa lista como una línea
CSV correcta — él se ocupa de las comillas y los escapes. **Eso no se reinventa a mano jamás.**

In [ ]:
%%bash
cd ../datasets
jq -r '.[] | [.id, .categoria, .precio] | @csv' productos.json | head -3

### Tiempo B · La versión de producción

Ahora con **cabecera** (una primera lista de literales) y un **campo calculado**: el stock total,
sumando los dos niveles del objeto `stock`.

In [ ]:
%%bash
cd ../datasets

jq -r '["id","nombre","categoria","precio","stock_total"],
       (.[] | [.id, .nombre, .categoria, .precio, (.stock.central + .stock.tiendas)])
       | @csv' productos.json > productos.csv

echo "== Las tres primeras líneas =="
head -3 productos.csv

echo ""
echo "== ¿Cuántas líneas tiene? =="
wc -l productos.csv

### ✅ LA VALIDACIÓN QUE DEFINE EL PASO

El número de líneas de `productos.csv` tiene que ser **el `length` del paso 1 más uno**: los 480
productos más la cabecera. **481.**

✍️ **¿Lo es?**


> Si no: compara tu comando con el del cuaderno **pieza a pieza y en voz alta**. Y recuerda que
> **la coma entre la cabecera y el bloque de filas no es decorativa**.

> 🎯 Acabas de hacer tu primera **transformación de formato** — la **T** del ETL en miniatura: entra
> jerarquía, sale tabla, y por el camino se calcula un dato que no existía. Spark hará esto mismo a
> escala industrial en la sesión 6. **La idea ya es tuya.**

---

## Paso 3 · ⚠️ LA TRAMPA DE LAS DOS PREGUNTAS

**Antes de ejecutar nada, responde.**

Pregunta de negocio: **¿cuántos productos no tienen stock?**

✍️ **Tu respuesta (un número):**

In [ ]:
%%bash
cd ../datasets

echo -n "Sin stock EN CENTRAL:        "
jq '[.[] | select(.stock.central == 0)] | length' productos.json

echo -n "Sin stock EN NINGÚN SITIO:   "
jq '[.[] | select(.stock.central + .stock.tiendas == 0)] | length' productos.json

# Esperado: 25 y 11

### 💡 Veinticinco y once

**Las dos son correctas.** Las dos responden a la misma pregunta.

El problema no estaba en jq: **estaba en la pregunta**. «Sin stock» no significa nada hasta que
alguien dice **dónde**.

✍️ **¿Cuál de las dos habrías entregado si te lo pide tu jefe? ¿Qué le habrías preguntado antes?**


> Y esto no es una curiosidad de laboratorio: es lo que te va a pasar con un compañero, con un
> cliente y —a partir de la sesión 7— **con la IA**, donde precisar el encargo tendrá nombre técnico
> y se llamará *ingeniería de prompts*.
>
> **Es la lección más barata y más rentable del curso.**

---

## Paso 4 · El ojo estadístico

In [ ]:
%%bash
cd ../datasets
jq -r '.[].categoria' productos.json | sort | uniq -c

✍️ **¿Te parece normal ese reparto? ¿Por qué sí o por qué no?**


> 💡 **No lo es.** Noventa y seis exactos en cada una de las cinco categorías. Los datos reales
> tienen ruido; **la uniformidad perfecta delata datos sintéticos**.
>
> Te lo digo con toda honestidad porque ese olfato —*«esto está demasiado redondo»*— es el mismo que
> luego detecta una métrica inflada o una respuesta de IA demasiado limpia.

---

# 🔍 CONSULTA · Bloque J

---

**J1 · 🗂️ RAG · BASE**

> *Según el manual del curso, ¿por qué `productos.json` se inspecciona con `head -c 400` y no con
> `head -3`? Cita el apartado.*

✍️


---

**J2 · 🤖 ASISTENTE · BASE**

> *¿Por qué `jq` necesita la bandera `-r` para producir CSV? ¿Qué sale exactamente sin ella?*

**Y ahora pruébalo tú**, quitando el `-r` de la celda del Tiempo A. No te fíes de su explicación sin
ejecutarla.

✍️


---

**J3 · ⚙️ MÁQUINA · BASE**

✍️ **¿Cuántos productos hay, cuántas líneas tiene `productos.csv`, y cuadra la cuenta?**


---

**J4 · 🗂️ RAG · BASE**

> *¿Qué hace `@csv` y por qué el material dice que las comillas y los escapes «no se reinventan a
> mano jamás»?*

✍️


---

**J5 · 📝 CRITERIO · BASE**

✍️ **`.stock.central` baja dos niveles de jerarquía. Escribe con tus palabras por qué un CSV no
puede expresar eso.**


---

**J6 · 🤖 ASISTENTE · COMPLETA — *y aquí hay trampa***

Pídele una variante:

> *Añade al CSV una columna extra que valga «SIN_STOCK» cuando el stock **total** sea 0, y «OK» en
> el resto. En jq existe el `if-then-else`.*

**Audita su respuesta con el protocolo de cuatro pasos antes de ejecutarla.** Y presta atención
especial a una cosa:

✍️ **¿Qué stock evalúa en el `if`: el total, como le pediste, o solo el central?**


> 💡 Si su salida marca SIN_STOCK a **25** productos en vez de a **11**, acabas de cazar a tu
> asistente **mezclando criterios**. Apúntate el tanto: **eso ES auditar.**

---
---

# LAB05 · Almacenamiento: HDFS y Parquet

Dos generaciones del almacenamiento en un solo laboratorio: **el distribuido clásico**, que es
cultura técnica viva, y **el formato que define el presente**, que vas a medir con tus propios datos.

---

## Parte 1 · HDFS — la demo del docente

### ⚠️ Esto NO se ejecuta desde el cuaderno

El comando `hdfs` vive dentro del contenedor `namenode`, y a los contenedores se les habla con
`docker exec` **desde el host**. Ni este cuaderno ni la Terminal de JupyterLab pueden hacerlo:
**ambos están dentro del contenedor de Jupyter, y un contenedor no gobierna a sus hermanos.**

Sigue la demo en el proyector. Los comandos, para tu bitácora:

```bash
# ⌂ HOST — terminal ssh del puesto
cd ~/curso-bigdata-ia

docker exec namenode hdfs dfs -mkdir -p /curso/datos
docker exec namenode hdfs dfs -put /datos/ventas.csv /curso/datos/
docker exec namenode hdfs dfs -ls -h /curso/datos
docker exec namenode hdfs fsck /curso/datos/ventas.csv
```

| Pieza | Qué hace |
|---|---|
| `docker exec namenode` | «ejecuta esto **dentro** del contenedor namenode» |
| `hdfs dfs` | El prefijo de todos los comandos de ficheros de HDFS |
| `-mkdir -p` · `-put` · `-ls -h` | **Calcan a los de Linux** — esa familiaridad fue parte del éxito de HDFS |
| `hdfs fsck` | La radiografía: **bloques y réplicas**. El mapa del NameNode, hecho visible |

Y el panel web: **`http://IP-DE-LA-VM:9870`** → *Utilities → Browse the file system*

### ✍️ Anota lo que veas

**¿Cuántos bloques tiene `ventas.csv` en HDFS? ¿Y qué factor de replicación?**


> 💡 60 MB es **menor** que el bloque por defecto de 128 MB, así que cabe en **uno solo**. Y la
> réplica será **1**, o verás bloques marcados como *under-replicated*.
>
> **Eso no es un fallo: es la demostración en negativo de la idea.** No puede haber tres copias en
> tres máquinas si solo hay una máquina. En producción, este mismo comando mostraría factor 3.

---

# 🔍 CONSULTA · Bloque K

**K1 · 🗂️ RAG · BASE**

> *Según el manual, ¿qué papel tiene el NameNode y qué papel el DataNode? ¿Qué pasa si se cae cada
> uno? Cita el apartado.*

✍️


---

**K2 · 🗂️ RAG · BASE**

> *¿Por qué el bloque de HDFS es de 128 MB y no del tamaño de un bloque de disco normal?*

✍️


---

**K3 · 📝 CRITERIO · COMPLETA**

✍️ **El manual dice que casi nadie monta un clúster Hadoop nuevo hoy. Entonces, ¿por qué crees que
te lo estamos enseñando?**


> *(pista: la respuesta está en el manual, y tiene que ver con las ofertas de empleo)*

---

## Parte 2 · Parquet — y ahora sí, con las manos

### Por filas contra por columnas

```
CSV (por filas):              Parquet (por columnas):
1,2025-01-02,…,Zaragoza       fechas:   [2025-01-02, 2025-01-02, 2025-01-03, …]
2,2025-01-02,…,Huesca         precios:  [219.90, 34.50, 89.90, …]
3,2025-01-03,…,Zaragoza       ciudades: [Zaragoza, Huesca, Zaragoza, …]
```

La analítica casi nunca quiere **registros completos**: quiere **columnas**. «Suma de precios por
categoría» toca dos columnas de nueve. Con un CSV no hay elección: para llegar a la quinta coma hay
que leer las cuatro anteriores, línea por línea.

**Parquet le da la vuelta.** Y de ese giro salen las dos ventajas que vas a medir ahora mismo.

---

## Paso 1 · Convertir

Le pedimos a **DuckDB** —el motor SQL del curso, aquí en su estreno— que lea el CSV y lo escriba en
formato columnar.

In [ ]:
import duckdb

# Si tu máquina va justa de memoria, descomenta la línea siguiente:
# duckdb.sql("SET memory_limit='512MB'")

duckdb.sql("COPY (SELECT * FROM '../datasets/ventas.csv') TO '../datasets/ventas.parquet' (FORMAT PARQUET)")
print("Convertido.")

> 🎯 Fíjate en el patrón: **original intacto, resultado en archivo nuevo.**
>
> Es la **primera ley de la ingeniería de datos** que viste en la teoría — y con Parquet deja de ser
> una buena práctica para convertirse en **la única forma posible de trabajar**: un binario no se
> edita, se genera uno nuevo.

---

## Paso 2 · Medir el tamaño

In [ ]:
%%bash
cd ../datasets

echo "== Tamaños =="
ls -l ventas.csv ventas.parquet | awk '{printf "  %-18s %12d bytes  (%.1f MiB)\n", $9, $5, $5/1048576}'

echo ""
echo "== La ratio =="
CSV=$(wc -c < ventas.csv)
PARQ=$(wc -c < ventas.parquet)
awk -v c=$CSV -v p=$PARQ 'BEGIN {printf "  %.2f veces más pequeño, sin perder un solo dato\n", c/p}'

# Esperado: CSV 61.942.187 B (59,1 MiB) · Parquet 17.178.881 B (16,4 MiB) · ratio ~3,6x

### ✍️ Anota los tres números

**Esa fila es el corazón del entregable.**

| | Bytes | MiB |
|---|---|---|
| `ventas.csv` | | |
| `ventas.parquet` | | |
| **Ratio** | | |

---

## Paso 3 · Medir la velocidad

### ⚠️ La mini-lección de método, antes de medir

**Ejecuta cada consulta dos veces y quédate con la segunda.** La primera paga el caché de disco, y
comparar una primera ejecución con una segunda **es hacerse trampas**.

Los tiempos absolutos variarán entre puestos. **Lo que tiene que salir con claridad es la ratio.**

In [ ]:
%time duckdb.sql("SELECT categoria, SUM(unidades*precio_unitario) FROM '../datasets/ventas.csv' GROUP BY categoria ORDER BY 2 DESC").show()

In [ ]:
%time duckdb.sql("SELECT categoria, SUM(unidades*precio_unitario) FROM '../datasets/ventas.csv' GROUP BY categoria ORDER BY 2 DESC").show()

### 🔍 Tres cosas que mirar en esa salida

---

**① El orden.** `informatica` arriba con **256,5 M€** y `papeleria` abajo con **9,3 M€**. ¿Te suena?
Es **exactamente el gráfico de la derecha de ayer** — la torre y los cuatro tocones. Mismo dato,
tercera herramienta.

---

**② `CPU times` contra `Wall time`.** Mira los dos números que imprime `%time`:

```
CPU times: user 273 ms, sys: 35.5 ms, total: 309 ms
Wall time: 168 ms
```

**El tiempo de CPU es MAYOR que el de reloj.** Eso solo puede significar una cosa: **han trabajado
varios núcleos a la vez**. No puedes gastar 309 milisegundos de procesador en 168 milisegundos de
reloj si solo hay un procesador trabajando.

✍️ **Calcula tú la proporción `CPU total / Wall` en tus cuatro ejecuciones. ¿Cuántos núcleos
diría que ha usado?**


---

**③ Los decimales.** Y aquí viene lo bueno. Mira el último decimal de `informatica` en **las dos
ejecuciones de la misma consulta**:

✍️ **¿Son idénticos, o cambian?**


> 💡 Si cambian, **no es un error de DuckDB ni tuyo**. En coma flotante **la suma no es asociativa**:
> agrupar los sumandos de otra manera da un resultado ligeramente distinto. Pruébalo:
>
> ```
> (0.1 + 0.2) + 0.3  →  0.6000000000000001
> 0.1 + (0.2 + 0.3)  →  0.6
> ```
>
> Y como acabas de descubrir en el punto ②, DuckDB **reparte el millón de filas entre varios
> hilos**: cada uno suma su trozo, y los parciales se combinan **en el orden en que van terminando**,
> que no está garantizado. De ahí que el último decimal baile.

> ⚠️ **La consecuencia, que es lo que importa:** sin redondear, **dos personas con los mismos datos y
> la misma consulta pueden llevar números distintos a una reunión** — no porque una se haya
> equivocado, sino porque el orden de las sumas cambió.
>
> Por eso la regla del curso no es estética: **se calcula con lo que hay y se presenta SIEMPRE
> redondeado a dos decimales.** El lunes le pondremos nombre y política.

In [ ]:
# Compruébalo tú: la suma en coma flotante NO es asociativa
print(f"(0.1 + 0.2) + 0.3 = {(0.1 + 0.2) + 0.3!r}")
print(f"0.1 + (0.2 + 0.3) = {0.1 + (0.2 + 0.3)!r}")
print(f"¿Son iguales?       {(0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3)}")
print()
print("Mismos números. Distinto orden de agrupación. Distinto resultado.")
print("Ahora imagina un millón de sumandos repartidos entre varios hilos.")

In [ ]:
%time duckdb.sql("SELECT categoria, SUM(unidades*precio_unitario) FROM '../datasets/ventas.parquet' GROUP BY categoria ORDER BY 2 DESC").show()

In [ ]:
%time duckdb.sql("SELECT categoria, SUM(unidades*precio_unitario) FROM '../datasets/ventas.parquet' GROUP BY categoria ORDER BY 2 DESC").show()

### 🔍 La pregunta buena

La diferencia de velocidad tiene **dos** causas. Una es la compresión.

✍️ **¿Cuál es la otra?** *(pista: cuenta cuántas columnas necesita esta consulta y cuántas hay que
leer en cada formato)*


---

## Paso 4 · La compresión es sin pérdida — demuéstralo

In [ ]:
duckdb.sql("SELECT COUNT(*) AS filas FROM '../datasets/ventas.parquet'").show()

# Esperado: 1000000
# Un millón. Compresión SIN PÉRDIDA, demostrada.

> 🎉 Y de paso: **acabas de ejecutar tu primer SQL del curso**, un día antes del módulo de SQL.
>
> Ese `SELECT COUNT(*)` es tu `wc -l` de ayer. El lunes le pondremos nombre a todo lo demás.

---

# 🔍 CONSULTA · Bloque P

---

**P1 · 🗂️ RAG · BASE**

> *Según el manual, ¿qué es la «codificación por diccionario» y por qué una columna de ciudades con
> diez valores distintos comprime tanto? Cita el apartado.*

✍️


---

**P2 · ⚙️ MÁQUINA · BASE**

✍️ **Tu ratio exacto, con los dos tamaños en bytes:**


---

**P3 · 📝 CRITERIO · BASE**

✍️ **Explica la diferencia de velocidad usando la palabra «columnar». Dos frases.**


---

**P4 · 🗂️ RAG · COMPLETA**

> *Un Parquet es binario: no se puede abrir con `nano` ni transformar con `sed`. Según el material,
> ¿cuál es la traducción de `head -5`, `wc -l` y «mirar la cabecera» al mundo Parquet?*

✍️


---

**P5 · ⚠️ LA TRAMPA · BASE**

Pregúntale a tu asistente, **sin darle ningún dato**:

> *Tengo un CSV de ventas de 60 MB. ¿Cuánto va a ocupar exactamente al convertirlo a Parquet?*

✍️ **¿Qué te ha contestado? ¿Ha dado un número exacto?**


> 💡 **No puede saberlo.** La compresión depende de cuántos valores repetidos hay en cada columna —
> y eso solo lo sabe **midiendo tu fichero**. Si te da un rango razonado *(«entre 3 y 5 veces menos,
> depende de la cardinalidad»)*, es una buena respuesta. Si te da una cifra exacta, **está
> inventada**.
>
> Segunda vez esta semana: **estimar no es calcular.**

---

**P6 · 📝 CRITERIO · COMPLETA**

El manual dice: *«CSV es formato de intercambio; Parquet es formato de trabajo analítico»*.

✍️ **Dame dos situaciones concretas: una en la que elegirías CSV aunque Parquet sea más rápido, y
otra en la que Parquet es obligatorio.**

---
---

# 📦 Entregable del bloque 1

**`Ctrl+S` antes de archivar.** La celda copia el fichero **guardado en disco**, no lo que ves en
pantalla.

### La tabla comparativa — el entregable oficial del LAB05

Rellénala con **tus** números:

| Formato | Tamaño (bytes) | Tamaño (MiB) | Tiempo de la consulta (2ª ejecución) |
|---|---|---|---|
| `ventas.csv` | | | |
| `ventas.parquet` | | | |
| **Ratio** | | | |

✍️ **Y una frase explicando la diferencia de velocidad con la palabra «columnar»:**


### Lo que tiene que estar hecho

| # | Contenido | ¿Hecho? |
|---|---|---|
| 1 | **`productos.csv` validado** — 481 líneas contra los 480 del `length` | |
| 2 | **La trampa de las dos preguntas** — 25 y 11, y qué significa | |
| 3 | **HDFS** — bloques y factor de replicación anotados de la demo | |
| 4 | **La tabla comparativa** completa | |
| 5 | Los bloques **J, K y P** contestados | |
| 6 | **Las dos trampas** (J6 y P5): qué cazaste | |

In [ ]:
import shutil, os, glob, json

SESION = 2

# ══════════════════════════════════════════════════════════════════════════
#  GUARDIÁN · ¿está en el disco lo que ves en pantalla?
#
#  Esta celda copia el FICHERO DEL DISCO, no lo que tienes delante. Jupyter
#  guarda solo cada pocos minutos: si archivas antes de un Ctrl+S, entregas
#  el cuaderno SIN tus resultados y el HTML sale sin las gráficas.
# ══════════════════════════════════════════════════════════════════════════

def resultados_en_disco(ruta):
    try:
        nb = json.load(open(ruta, encoding="utf-8"))
    except Exception:
        return 0, 0
    codigo = [c for c in nb["cells"] if c["cell_type"] == "code"]
    return sum(1 for c in codigo if c.get("outputs")), len(codigo)

cuadernos = [f for f in glob.glob("*lab04*.ipynb") if ".ipynb_checkpoints" not in f]
listo = bool(cuadernos)

if not cuadernos:
    print("  No encuentro el cuaderno de este laboratorio en esta carpeta.")

for cuaderno in cuadernos:
    hechas, total = resultados_en_disco(cuaderno)
    print(f"  {cuaderno}: {hechas} de {total} celdas con resultados en el disco")
    if hechas == 0:
        listo = False

if not listo:
    print()
    print("  " + "=" * 68)
    print("   PARA AQUI. No he archivado nada.")
    print()
    print("   Pulsa  Ctrl+S  (Cmd+S en Mac)  y vuelve a ejecutar ESTA celda.")
    print("   Se archiva el fichero del DISCO, no lo que ves en pantalla.")
    print("  " + "=" * 68)

else:
    os.makedirs("entregables", exist_ok=True)

    # productos.csv es PIEZA OFICIAL DEL PROYECTO: se une a las ventas el martes
    piezas = cuadernos + [f for f in ["mi_bitacora.ipynb", "../datasets/productos.csv"] if os.path.exists(f)]

    print()
    for pieza in piezas:
        shutil.copy(pieza, f"entregables/S{SESION:02d}_{os.path.basename(pieza)}")
        print("  copiado:", pieza)

    anidada = os.path.join("entregables", "entregables")
    if os.path.isdir(anidada):
        shutil.rmtree(anidada)
        print("  limpiado: entregables anidado de una ejecución anterior")

    print()
    print("Contenido de entregables/:")
    for f in sorted(os.listdir("entregables")):
        if f != ".ipynb_checkpoints":
            kb = os.path.getsize(os.path.join("entregables", f)) / 1024
            print(f"    {f:<34} {kb:>8.0f} KB")

In [ ]:
import glob, os

# Exporta a HTML el cuaderno de este laboratorio. El HTML conserva las
# gráficas y las salidas incrustadas: se manda por correo y se ve entero.
for cuaderno in glob.glob("*lab04*.ipynb"):
    if ".ipynb_checkpoints" in cuaderno:
        continue
    salida = f"S{SESION:02d}_" + os.path.splitext(os.path.basename(cuaderno))[0]
    !jupyter nbconvert --to html --output-dir entregables --output {salida} "{cuaderno}"

In [ ]:
import glob, os, re

# ══════════════════════════════════════════════════════════════════════════
#  LA COMPROBACIÓN QUE CIERRA EL CÍRCULO
#
#  Un cuaderno ejecutado deja en el HTML el número de cada celda: [1]:, [2]:...
#  Si no hay ninguno, el HTML NO lleva tus resultados.
#  El TAMAÑO del fichero engaña; este número, no.
# ══════════════════════════════════════════════════════════════════════════

def celdas_ejecutadas(ruta_html):
    h = open(ruta_html, encoding="utf-8", errors="ignore").read()
    return len(re.findall(r"\[[0-9]+\]:", h)), h.count("data:image/png;base64")

htmls = sorted(glob.glob("entregables/*.html"))
vacios = []

if not htmls:
    print("  No hay ningún HTML. ¿Ejecutaste la celda anterior?")

for h in htmls:
    ejecutadas, graficas = celdas_ejecutadas(h)
    kb = os.path.getsize(h) / 1024
    if ejecutadas == 0:
        vacios.append(os.path.basename(h))
    estado = "OK    " if ejecutadas else "VACIO "
    print(f"  {estado} {os.path.basename(h):<32} {ejecutadas:>3} celdas ejecutadas · "
          f"{graficas} gráfica(s) · {kb:.0f} KB")

print()
if not htmls:
    print("  Vuelve a la celda de archivar: sin ella no hay nada que exportar.")
elif vacios:
    print("  " + "=" * 68)
    print("   ESE HTML NO LLEVA TUS RESULTADOS.")
    print("   Pulsa Ctrl+S y repite las DOS celdas anteriores.")
    print("  " + "=" * 68)
else:
    print("  Tu entrega lleva tus resultados. Puedes empaquetarla.")

### El paquete del bloque, para Moodle

In [ ]:
import os, re, glob, zipfile

APELLIDO_NOMBRE = "PEREZ_Ana"        # <-- pon el tuyo

# Última verificación antes de empaquetar: ningún HTML puede estar vacío.
vacios = [os.path.basename(h) for h in glob.glob("entregables/*.html")
          if not re.findall(r"\[[0-9]+\]:", open(h, encoding="utf-8", errors="ignore").read())]

if vacios:
    print("  " + "=" * 68)
    print("   NO EMPAQUETO: estos HTML no llevan resultados.")
    for v in vacios:
        print("     -", v)
    print("   Pulsa Ctrl+S y repite las tres celdas anteriores.")
    print("  " + "=" * 68)

else:
    nombre_zip = f"{APELLIDO_NOMBRE}_bloque1.zip"

    # A mano en vez de con make_archive, para poder EXCLUIR los checkpoints
    # de Jupyter: si no, se cuelan y multiplican el tamaño de la entrega.
    with zipfile.ZipFile(nombre_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for raiz, carpetas, ficheros in os.walk("entregables"):
            carpetas[:] = [c for c in carpetas if c != ".ipynb_checkpoints"]
            for f in ficheros:
                ruta = os.path.join(raiz, f)
                z.write(ruta, os.path.relpath(ruta, "entregables"))

    print(f"Creado: {nombre_zip}  ({os.path.getsize(nombre_zip)/1024:.0f} KB)")
    print()
    print("Contiene:")
    with zipfile.ZipFile(nombre_zip) as z:
        for n in sorted(z.namelist()):
            print("   ", n)
    print()
    print("Ahora: panel de archivos de JupyterLab -> clic derecho sobre el .zip -> Download")
    print("Y de ahi, a la tarea de Moodle.")

---
---

# 🏁 Cierre del Bloque 1 · haz inventario

Es más de lo que parece.

- Un **entorno reproducible** que levantas con una orden y recuperas en cinco minutos.
- **Cuatro datasets** explorados, medidos y con su suciedad **censada**: sabes cuánto pesan, cuántos
  registros tienen y **dónde mienten**.
- Un cuadro de **métricas de negocio y de operaciones** calculado con tuberías, **verificado por dos
  vías**, y con un **ataque real detectado** dentro.
- El **catálogo transformado** de JSON a CSV con un campo calculado — tu primera **T** del ETL, y
  pieza oficial del proyecto.
- Las ventas en **Parquet**, con la **evidencia medida** de por qué ese formato domina la industria.
- Y un **asistente de IA** al que has aprendido a dirigir con contexto, a usar como red de rescate
  — y **a auditar cuando mezcla criterios**.

### En el idioma en que se escriben los currículums

> Exploración y perfilado de datos en línea de comandos · cálculo y verificación de indicadores
> sobre datasets de millones de registros · análisis de logs de servidor con detección de anomalías
> · transformación de formatos (JSON→CSV, CSV→Parquet) · fundamentos de arquitecturas de datos
> modernas · uso profesional de asistentes de IA con verificación sistemática de salidas.

**Escríbelo así, porque es literalmente lo que has hecho.**

---

## 📖 Para el lunes · lectura recomendada

La teoría te la explico en clase, como siempre. Pero el manual está escrito para leerse antes, y hay
una diferencia real entre **oír** una explicación por primera vez y **reconocerla**.

```
Manual del Bloque 2 · secciones 4.1 a 4.5

  4.1  El modelo relacional: por qué las tablas ganaron
  4.2  Ficha de herramienta: DuckDB
  4.3  Las seis cláusulas y su orden REAL de ejecución
  4.4  Tipos de datos y el baile del decimal
  4.5  LIMPIO-v1   <--  si solo lees una, que sea esta
```

Son unos veinte minutos. Y si no llegas, no pasa nada: el lunes lo vemos igual.

### Tres preguntas para ir pensando

No hace falta acertar — **fallarlas es lo que hace que se queden**.

- ¿`COUNT(*)` sobre `ventas.csv` dirá 1.000.000 o 1.000.001?
- Los censos de la suciedad (3030 · 2004 · 941 · 465), ¿saldrán idénticos en SQL?
- Al limpiar el fichero, la facturación —**429.888.864,70 €**— ¿subirá o bajará?

---

### El lunes: ya sabes SQL a medias

| Lo que hiciste en la terminal | Cómo se dice en SQL |
|---|---|
| `head -5 f.csv` | `SELECT * FROM 'f.csv' LIMIT 5` |
| `wc -l` | `SELECT COUNT(*)` |
| `cut -d, -f8` | `SELECT ciudad` |
| <code>sort &#124; uniq -c &#124; sort -rn</code> | `GROUP BY … ORDER BY n DESC` |
| `awk '{t+=$6*$7} END{print t}'` | `SUM(unidades*precio_unitario)` |
| <code>sort -u &#124; wc -l</code> | `COUNT(DISTINCT …)` |
| <code>jq '.[] &#124; select(.precio&gt;100)'</code> | `WHERE precio > 100` |

`GROUP BY` no va a ser una palabra mágica: va a ser **tu `sort | uniq -c` con traje**. Y esa es
exactamente la razón de que hayamos empezado por la terminal y no por SQL.

**Trae tu `productos.csv`: le tenemos preparado un JOIN con sorpresa.**